# RAM Antivirus — Member 3: API Attacks, Threat Correlation & Dynamic Risk
### **Assigned Scope:** C4 Suspicious API | C11 BOLA / SSRF | C13 Database Threats | C14 Serverless Abuse
**Author:** Member 3 — Threat Intelligence & Correlation  
**Supporting Engines:** `backend/engines/correlation_engine.py`, `risk_engine.py`, `simulator/`

---
## Overview & Objectives
Member 3 connects isolated security findings into comprehensive multi-stage security incidents using temporal sliding windows and computes unified dynamic risk scores.


In [ ]:
# %pip install -q pandas numpy matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("[OK] Member 3 workspace ready!")


---
## 1. Multi-Signal Incident Correlation (900-Second Sliding Window)
Findings sharing the same entity (`principal`, `source_ip`, or `target_resource`) within 900 seconds are correlated into a single security incident.


In [ ]:
findings = pd.DataFrame([
    {"finding_id": "F-01", "time": 0,   "principal": "attacker_ip_45", "stage": "Recon", "type": "Network Scan (C7)"},
    {"finding_id": "F-02", "time": 120, "principal": "attacker_ip_45", "stage": "Initial Access", "type": "BOLA / SSRF (C11)"},
    {"finding_id": "F-03", "time": 300, "principal": "attacker_ip_45", "stage": "Privilege Escalation", "type": "IAM PrivEsc (C3)"},
    {"finding_id": "F-04", "time": 600, "principal": "attacker_ip_45", "stage": "Exfiltration", "type": "Data Exfiltration (C6)"},
])

print("Correlated Incident Findings (linked by 'attacker_ip_45'):")
findings


---
## 2. MITRE Kill-Chain Progression Tracking
We track progression: $\text{Recon} \to \text{Initial Access} \to \text{Privilege Escalation} \to \text{Defense Evasion} \to \text{Exfiltration} \to \text{Impact}$.  
As an attack progresses across stages, incident severity multiplies.


In [ ]:
# Plotting Kill-Chain Progression Waterfall
stages = ["Recon", "Initial Access", "Privilege Escalation", "Exfiltration"]
times = [0, 2, 5, 10]

plt.figure(figsize=(9, 4))
plt.step(times, stages, where="post", color="#d9480f", linewidth=2.5, marker="o")
plt.title("Multi-Stage Attack Progression Over Time", fontweight="bold")
plt.xlabel("Elapsed Time (Minutes)")
plt.ylabel("MITRE Kill-Chain Stage")
plt.grid(True)
plt.tight_layout()
plt.show()


---
## 3. Unified Dynamic Risk Formula (4 Pillars)
$$\text{Composite Risk} = 0.35 \times \text{Incident} + 0.25 \times \text{ML Anomaly} + 0.20 \times \text{CSPM} + 0.20 \times \text{CIEM}$$


In [ ]:
def compute_risk(incident_score, ml_score, cspm_score, ciem_score):
    return (0.35 * incident_score) + (0.25 * ml_score) + (0.20 * cspm_score) + (0.20 * ciem_score)

# Example: Active multi-stage breach
risk = compute_risk(incident_score=90, ml_score=85, cspm_score=70, ciem_score=75)
print(f"Calculated Unified Composite Risk Score: {risk:.1f} / 100")


---
## 4. Member 3 Viva Cheat Sheet
- **Q1: Why correlate alerts?**  
  *Ans:* To prevent alert fatigue by grouping related findings into one coherent incident narrative.
- **Q2: Why a 900-second window?**  
  *Ans:* 15 minutes is the standard dwell time between automated reconnaissance and initial exploitation steps.
- **Q3: What are the 4 pillars of the risk score?**  
  *Ans:* Incident Severity (35%), ML Anomaly (25%), CSPM Posture (20%), and CIEM Entitlements (20%).
